In [2]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
from litellm import completion
from pydantic import BaseModel

from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:

import json
import time
from typing import Any

class LLMClient:
    def __init__(self, model_name: str, max_completion_tokens: int = 4000):
        self.model_name = model_name
        self.request_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_tokens_used = 0
        self.total_latency_ms = 0.0
        self.last_usage: dict[str, Any] = {}
        self.last_error: str | None = None
        self.last_response = None
        self.max_completion_tokens = max_completion_tokens

    def get_usage(self) -> str:
        return (
            f"Using model: {self.model_name}, Requests: {self.request_count}, "
            f"Prompt tokens: {self.total_prompt_tokens}, Completion tokens: {self.total_completion_tokens}, "
            f"Total tokens used: {self.total_tokens_used}, Total latency ms: {round(self.total_latency_ms, 2)}"
        )

    def get_usage_details(self) -> dict[str, Any]:
        return {
            "model_name": self.model_name,
            "request_count": self.request_count,
            "prompt_tokens": self.total_prompt_tokens,
            "completion_tokens": self.total_completion_tokens,
            "total_tokens": self.total_tokens_used,
            "total_latency_ms": round(self.total_latency_ms, 2),
            "last_usage": self.last_usage,
            "last_error": self.last_error,
        }

    def _record_usage(self, response: Any, latency_ms: float) -> None:
        self.request_count += 1
        self.total_latency_ms += latency_ms
        usage = getattr(response, "usage", None)
        if usage:
            prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
            completion_tokens = getattr(usage, "completion_tokens", 0) or 0
            total_tokens = getattr(usage, "total_tokens", prompt_tokens + completion_tokens) or 0
            self.total_prompt_tokens += prompt_tokens
            self.total_completion_tokens += completion_tokens
            self.total_tokens_used += total_tokens
            self.last_usage = {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens,
            }
        else:
            self.last_usage = {}

    def _extract_choice(self, response: Any) -> Any:
        choices = getattr(response, "choices", None)
        if choices and len(choices) > 0:
            return choices[0]
        return None

    def _extract_message(self, response: Any) -> Any:
        choice = self._extract_choice(response)
        return getattr(choice, "message", None) if choice else None

    def _extract_tool_calls(self, message: Any) -> list[dict[str, Any]]:
        tool_calls = getattr(message, "tool_calls", None) or []
        normalized_calls: list[dict[str, Any]] = []
        for tool_call in tool_calls:
            function_call = getattr(tool_call, "function", None)
            normalized_calls.append({
                "id": getattr(tool_call, "id", None),
                "type": getattr(tool_call, "type", None),
                "name": getattr(function_call, "name", None),
                "arguments": getattr(function_call, "arguments", None),
            })
        return normalized_calls

    def _parse_structured_content(self, content: str | None, response_format: type[BaseModel] | None) -> Any:
        if not content:
            return {}

        if response_format and isinstance(response_format, type) and issubclass(response_format, BaseModel):
            try:
                return response_format.model_validate_json(content)
            except Exception:
                pass

        try:
            return json.loads(content)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Structured response was not valid JSON: {exc}") from exc

    def generate(self, messages: list[dict[str, str]], max_completion_tokens: int | None = None, tools: list[dict[str, Any]] | None = None, tool_choice: Any = None, return_full_response: bool = False, temperature: float | None = 0) -> str | dict[str, Any]:
        resolved_max_completion_tokens = self.max_completion_tokens if max_completion_tokens is None else max_completion_tokens
        request_kwargs: dict[str, Any] = {
            "model": self.model_name,
            "messages": messages,
            "max_completion_tokens": resolved_max_completion_tokens,
            "stream": False,
            "temperature": temperature,
        }
        if tools is not None:
            request_kwargs["tools"] = tools
        if tool_choice is not None:
            request_kwargs["tool_choice"] = tool_choice

        start_time = time.perf_counter()
        self.last_error = None
        try:
            response = completion(**request_kwargs)
        except Exception as exc:
            self.last_error = str(exc)
            raise
        latency_ms = (time.perf_counter() - start_time) * 1000
        self.last_response = response
        self._record_usage(response, latency_ms)

        message = self._extract_message(response)
        content = getattr(message, "content", None) if message else None
        tool_calls = self._extract_tool_calls(message) if message else []
        text = content or ("No tool call content returned." if tool_calls else "No content in response.")

        if return_full_response:
            return {
                "content": content,
                "text": text,
                "tool_calls": tool_calls,
                "usage": self.last_usage,
                "latency_ms": round(latency_ms, 2),
                "model_name": self.model_name,
                "raw_response": response,
            }

        return text

    def generate_structured(self, messages: list[dict[str, str]], response_format: type[BaseModel] | None = None, max_completion_tokens: int| None = None, tools: list[dict[str, Any]] | None = None, tool_choice: Any = None, temperature: float | None = 0) -> Any:
        resolved_max_completion_tokens = self.max_completion_tokens if max_completion_tokens is None else max_completion_tokens
        request_kwargs: dict[str, Any] = {
            "model": self.model_name,
            "messages": messages,
            "max_completion_tokens": resolved_max_completion_tokens,
            "stream": False,
            "temperature": temperature,
        }
        if response_format is not None:
            request_kwargs["response_format"] = response_format
        if tools is not None:
            request_kwargs["tools"] = tools
        if tool_choice is not None:
            request_kwargs["tool_choice"] = tool_choice

        start_time = time.perf_counter()
        self.last_error = None
        try:
            response = completion(**request_kwargs)
        except Exception as exc:
            self.last_error = str(exc)
            raise
        latency_ms = (time.perf_counter() - start_time) * 1000
        self.last_response = response
        self._record_usage(response, latency_ms)

        message = self._extract_message(response)
        content = getattr(message, "content", None) if message else None
        if not content:
            return {}
        return self._parse_structured_content(content, response_format)
    

In [4]:

openai_client = LLMClient(model_name="openai/gpt-4o-mini", max_completion_tokens=4000)
groq_client = LLMClient(model_name="groq/openai/gpt-oss-120b", max_completion_tokens=4000)
github_client = LLMClient(model_name="github/Llama-3.3-70B-Instruct", max_completion_tokens=4000)
github_client_light = LLMClient(model_name="github/gpt-4o-mini", max_completion_tokens=2000)

In [5]:
llm_client = github_client

In [64]:
class TempResponse(BaseModel):
    answer: str
    confidence: float

res=github_client_light.generate_structured([{"role": "user", "content": "What is the capital of France?"}], response_format=TempResponse, max_completion_tokens=100)

In [65]:
res

TempResponse(answer='The capital of France is Paris.', confidence=1.0)

In [66]:
# llm_client.get_usage()

In [6]:
uri = "lancedb"
db = lancedb.connect(uri)

In [7]:
CONVERSATIONAL_TABLE = "CONVERSATIONAL_MEMORY"
KNOWLEDGE_BASE_TABLE = "SEMANTIC_MEMORY"
WORKFLOW_TABLE = "WORKFLOW_MEMORY"
TOOLBOX_TABLE = "TOOLBOX_MEMORY"
ENTITY_TABLE = "ENTITY_MEMORY"
SUMMARY_TABLE = "SUMMARY_MEMORY"
TOOL_LOG_TABLE = "TOOL_LOG_MEMORY"

In [8]:
func = get_registry().get("huggingface").create(name="colbert-ir/colbertv2.0")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
func.ndims()

768

In [10]:
ALL_TABLES = [
    CONVERSATIONAL_TABLE,
    KNOWLEDGE_BASE_TABLE,
    WORKFLOW_TABLE,
    TOOLBOX_TABLE,
    ENTITY_TABLE,
    SUMMARY_TABLE,
    TOOL_LOG_TABLE]

# clear all tables
for table in ALL_TABLES:
    try:
        db.drop_table(table)
    except Exception as e:
        print(f"Error dropping table {table}: {e}")

In [11]:
from typing import Optional
from datetime import datetime, timezone

from pydantic import Field


class ConversationLog(LanceModel):
    id: Optional[str] = None
    thread_id: str
    role: str
    content: str
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    metadata: Optional[str] = None
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    summary_id: Optional[str] = None


class ToolLog(LanceModel):
    id: Optional[str] = None
    thread_id: str
    tool_call_id: Optional[str] = None
    tool_name: str
    tool_args: Optional[str] = None
    result: Optional[str] = None
    result_preview: Optional[str] = None
    status: str = "success"
    error_message: Optional[str] = None
    metadata: Optional[str] = None
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class MetadataEntry(LanceModel):
    key: str
    value: str

class KnowledgeBaseEntry(LanceModel):
    id: Optional[str] = None
    thread_id: Optional[str] = None
    text: str = func.SourceField()
    vector: Optional[Vector(func.ndims())] = func.VectorField()  # pyright: ignore[reportInvalidTypeForm]
    metadata: Optional[list[MetadataEntry]] = None
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class Entity(BaseModel):
    name: str
    type: str
    description: Optional[str] = None

class EntityExtractionResponse(BaseModel):
    entities: list[Entity]

In [ ]:
class StorageManger:
    """
    Responsible for creating tables and setting up getters for each table.
    """
    def __init__(self, db_uri: str):
        self.db = lancedb.connect(db_uri)
        self.conversation_table = db.create_table(CONVERSATIONAL_TABLE, schema=ConversationLog, mode="overwrite")
        self.tool_log_table = db.create_table(TOOL_LOG_TABLE, schema=ToolLog, mode="overwrite")
        self.knowledge_base_table = db.create_table(KNOWLEDGE_BASE_TABLE, schema=KnowledgeBaseEntry, mode="overwrite")
        self.workflow_table = db.create_table(WORKFLOW_TABLE, schema=KnowledgeBaseEntry, mode="overwrite")
        self.toolbox_table = db.create_table(TOOLBOX_TABLE, schema=KnowledgeBaseEntry, mode="overwrite")
        self.entity_table = db.create_table(ENTITY_TABLE, schema=KnowledgeBaseEntry,  mode="overwrite")
        self.summary_table = db.create_table(SUMMARY_TABLE, schema=KnowledgeBaseEntry, mode="overwrite")

    def get_conversation_table(self):
        return self.db.open_table(CONVERSATIONAL_TABLE)
    
    def get_tool_log_table(self):
        return self.db.open_table(TOOL_LOG_TABLE)
    
    def get_knowledge_base_table(self):
        return self.db.open_table(KNOWLEDGE_BASE_TABLE)
    
    def get_workflow_table(self):
        return self.db.open_table(WORKFLOW_TABLE)
    
    def get_toolbox_table(self):
        return self.db.open_table(TOOLBOX_TABLE)
    
    def get_entity_table(self):
        return self.db.open_table(ENTITY_TABLE)
    
    def get_summary_table(self):
        return self.db.open_table(SUMMARY_TABLE)
    

class MemoryManager:
    """
    Responsible for managing memory operations such as adding entries, retrieving entries, and performing similarity searches.
    """
    def __init__(self, storage_manager: StorageManger):
        self.storage_manager = storage_manager
        self.conversation_table = storage_manager.get_conversation_table()
        self.tool_log_table = storage_manager.get_tool_log_table()
        self.knowledge_base_table = storage_manager.get_knowledge_base_table()
        self.workflow_table = storage_manager.get_workflow_table()
        self.toolbox_table = storage_manager.get_toolbox_table()
        self.entity_table = storage_manager.get_entity_table()
        self.summary_table = storage_manager.get_summary_table()

    def add_conversation_log(self, log: list[ConversationLog]):
        self.conversation_table.add(log)

    def add_tool_log(self, log: list[ToolLog]):
        self.tool_log_table.add(log)
    
    def add_knowledge_base_entry(self, entry: list[KnowledgeBaseEntry]):
        self.knowledge_base_table.add(entry)
    
    def add_workflow_entry(self, entry: list[KnowledgeBaseEntry]):
        self.workflow_table.add(entry)
    
    def add_toolbox_entry(self, entry:  list[KnowledgeBaseEntry]):
        self.toolbox_table.add(entry)

    def add_summary_entry(self, entry:  list[KnowledgeBaseEntry]):
        self.summary_table.add(entry)

    def read_toolbox_entry(self, query, limit=5):
        results = self.toolbox_table.search(
            query=query,
            query_type="vector",
            vector_column_name="vector",
        ).distance_type('cosine').to_pandas().sort_values("_distance").head(limit)
        # filter only when distance is less than a threshold (e.g., 0.2 for cosine similarity)
        results = results[results["_distance"] < 0.5]

        tools=[] # should contain all the tool names

        for _, row in results.iterrows():
            metadata_list = row.get("metadata") or []
            metadata_dict = {}
            for entry in metadata_list:
                if isinstance(entry, dict):
                    key = entry.get("key")
                    value = entry.get("value")
                else:
                    key = getattr(entry, "key", None)
                    value = getattr(entry, "value", None)
                if key is not None:
                    metadata_dict[key] = value

            tool_name = metadata_dict.get("tool_name")
            if tool_name:
                tools.append(tool_name)

        return tools
            

        # get tool functions 

    def add_entity_entry(self, query):

        prompt = f"Extract the entities from the following text and provide them in a list of JSON objects with 'name', 'type', and 'description' (if available):\n\n{query}"
        response = llm_client.generate_structured(
            messages=[{"role": "user", "content": prompt}],
            response_format= EntityExtractionResponse,
            max_completion_tokens=500
        )

        if isinstance(response, EntityExtractionResponse):
            entities = response.entities
            entries = []
            for entity in entities:
                entry = KnowledgeBaseEntry(
                    text=f"Name : {entity.name}, Type: {entity.type}, Description: {entity.description or 'N/A'}",
                    metadata=[MetadataEntry(key="name", value=entity.name), MetadataEntry(key="type", value=entity.type), MetadataEntry(key="description", value=entity.description or "")]
                )
                entries.append(entry)
            self.entity_table.add(entries)

    def read_knowledge_base_entry(self, query, limit=5)-> str:
        data = self.knowledge_base_table.search(
            query=query,
            query_type="vector",
            vector_column_name="vector",
        ).to_pandas().sort_values("_distance").head(limit)
        if data.empty:
            return "No relevant information found in the knowledge base."
        content = "\n\n".join(data["text"].tolist())
        return f"""## Knowledge Base Memory
        ### What this memory is
        Retrieved background documents and previously ingested reference material relevant to the current query.
        ### How you should leverage it
        - Ground responses in these passages when making factual or technical claims.
        - Prefer concrete details from this memory over unsupported assumptions.
        - If evidence is missing or ambiguous, state uncertainty and request clarification or additional retrieval.
        ### Retrieved passages

        {content}"""


    def read_conversation_logs(self, thread_id: str) -> str:
        data = self.conversation_table.search().where("(thread_id = 'thread1') & (summary_id is null)").to_pandas().sort_values("timestamp")
        messages = [f"{row['timestamp'].strftime('%H:%M:%S')} {row['role']}: {row['content']}" for _, row in data.iterrows()]
        return f"""## Conversation Memory
            ### What this memory is
            Chronological, unsummarized messages from the current thread. This memory captures user intent, constraints, and commitments made in recent turns.
            ### How you should leverage it
            - Preserve continuity with prior decisions, terminology, and user preferences.
            - Resolve references like "that", "previous step", or "the paper above" using earlier turns.
            - If older context conflicts with newer user instructions, prioritize the latest user direction.
            ### Retrieved messages
            
            {messages}"""

    def read_entity_entry(self, query, limit=3):
        res = self.entity_table.search(
            query=query,
            query_type="vector",
            vector_column_name="vector",
        ).to_pandas().sort_values("_distance").head(limit)["text"].tolist()

        stri = f"""## Entity Memory
            ### What this memory is
            Entity-level context such as people, organizations, systems, tools, and other named items previously identified in conversations or documents.
            ### How you should leverage it
            - Use entities to disambiguate references and maintain consistent naming.
            - Preserve important attributes (roles, relationships, descriptions) across turns.
            - Personalize and contextualize responses using relevant known entities.
            ### Retrieved entities

        {res}"""

        return stri

    def read_workflow_memory(self, query):
        res = self.workflow_table.search(
            query=query,
            query_type="vector",
            vector_column_name="vector",
        ).to_pandas().sort_values("_distance").head(5)["text"].tolist()

        if not res:
            return """## Workflow Memory
### What this memory is
Past task trajectories that include query context, ordered steps taken, and prior outcomes.
### How you should leverage it
- Use these workflows as reusable execution patterns for planning and tool orchestration.
- Adapt step sequences to the current task rather than copying blindly.
- Reuse successful patterns first, then adjust when task scope or constraints differ.
### Retrieved workflows
(No relevant workflows found.)"""
        content = "\n\n".join(res)
        return f"""## Workflow Memory
### What this memory is
Past task trajectories that include query context, ordered steps taken, and prior outcomes.
### How you should leverage it
- Use these workflows as reusable execution patterns for planning and tool orchestration.
- Adapt step sequences to the current task rather than copying blindly.
- Reuse successful patterns first, then adjust when task scope or constraints differ.
### Retrieved workflows

{content}"""









class SummaryManager:
    def __init__(self, storage_manager: StorageManger):
        self.summary_table = storage_manager.get_summary_table()

    


storage_manager = StorageManger(uri)
memory_manager = MemoryManager(storage_manager)
summary_manager = SummaryManager(storage_manager)



[2026-04-14T20:33:55Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/CONVERSATIONAL_MEMORY.lance, it will be created
[2026-04-14T20:33:55Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/TOOL_LOG_MEMORY.lance, it will be created
[2026-04-14T20:33:55Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/SEMANTIC_MEMORY.lance, it will be created
[2026-04-14T20:33:55Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/WORKFLOW_MEMORY.lance, it will be created
[2026-04-14T20:33:55Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/TOOLBOX_MEMORY.lance, it will be created
[2026-04-14T20:33:55Z WAR

In [87]:
import inspect
import json
from typing import Type, Callable, Any, Dict
from pydantic import BaseModel
from openai.lib import pydantic_function_tool

class ToolRegistry:
    def __init__(self, client: LLMClient, memory_manager: MemoryManager):
        self.client = client
        self.memory_manager = memory_manager
        self.tools: Dict[str, Dict[str, Any]] = {}

    def register(self, param_schema: Type[BaseModel]):
        def decorator(func: Callable):
            func_name = func.__name__
            
            openai_tool_schema = pydantic_function_tool(param_schema)
            
            # Update the schema to use the actual function name instead of the param class name
            openai_tool_schema["function"]["name"] = func_name
            
            source_code = inspect.getsource(func)
            
            self.tools[func_name] = {
                "callable": func,
                "schema": openai_tool_schema,
                "param_model": param_schema,
                "source_code": source_code,
                "augmented_metadata": None  # Populated in next step
            }


            return func
        return decorator
    
    def _save_tool_to_db(self, tool_name: str):
        tool_data = self.tools[tool_name]
        build_string = f"Tool: {tool_name}\n\nDescription: {tool_data['augmented_metadata']['search_doc']}\n\nTrigger Queries:\n"
        for query in tool_data["augmented_metadata"]["trigger_queries"]:
            build_string += f"- {query}\n"
        entry = KnowledgeBaseEntry(
            text=build_string,
            metadata=[MetadataEntry(key="tool_name", value=tool_name)],
            vector=None
        )
        self.memory_manager.add_toolbox_entry([entry])

    def augment_all(self):
        """Iterates through registered tools and enriches them via LLM."""
        for name, data in self.tools.items():
            print(f"Augmenting {name}...")
            self.tools[name]["augmented_metadata"] = self._get_llm_augmentation(data)

    def save_all_tools_to_db(self):
        """Saves all tools with their augmented metadata to the database."""
        for name in self.tools.keys():
            self._save_tool_to_db(name)

    def _get_llm_augmentation(self, tool_data: Dict) -> Dict:
        prompt = f"""
        Analyze this Python function and its Pydantic parameter schema.
        Generate a detailed 'search_doc' that includes:
        1. A comprehensive description of WHAT it does.
        2. WHEN it should be used (context).
        3. 5 diverse user queries that would trigger this tool.
        
        CODE:
        {tool_data['source_code']}
        
        SCHEMA:
        {json.dumps(tool_data['schema'], indent=2)}
        """
        
        class ToolMetadata(BaseModel):
            search_doc: str
            trigger_queries: list[str]

        response = self.client.generate_structured(
            messages=[{"role": "system", "content": "You are a technical documentation assistant."},
                      {"role": "user", "content": prompt}],
            response_format=ToolMetadata
            
        )
        print("TempResponse from LLM:", response)
        return response.model_dump()


In [88]:
registry = ToolRegistry(client=github_client_light, memory_manager=memory_manager)

class OrderParams(BaseModel):
    symbol: str
    qty: int

@registry.register(OrderParams)
def place_market_order(symbol: str, qty: int):
    """Executes a market order on the exchange."""
    # Your trading logic here
    return f"Order placed for {qty} shares of {symbol}"

# generate some sample tools to populate the registry
class WeatherParams(BaseModel):
    location: str

@registry.register(WeatherParams)
def get_weather(location: str):
    """Fetches current weather for a given location."""
    # Your weather fetching logic here
    return f"Current weather in {location} is Sunny, 25°C"

class NewsParams(BaseModel):
    topic: str


@registry.register(NewsParams)
def get_news(topic: str):
    """Retrieves latest news articles on a specified topic."""
    # Your news fetching logic here
    return f"Latest news on {topic}: 'AI breakthrough in natural language processing!'"


class StockParams(BaseModel):
    ticker: str
    
@registry.register(StockParams)
def get_stock_price(ticker: str):
    """Gets the current stock price for a given ticker symbol."""
    # Your stock price fetching logic here
    return f"The current stock price of {ticker} is $150.00"    


class RecipeParams(BaseModel):
    dish: str

@registry.register(RecipeParams)
def get_recipe(dish: str):
    """Provides a recipe for a specified dish."""
    # Your recipe fetching logic here
    return f"Recipe for {dish}: 1. Gather ingredients... 2. Cook... 3. Serve!"

# Run the augmentation pipeline
registry.augment_all()
# registry.save_all_tools_to_db()

Augmenting place_market_order...
TempResponse from LLM: search_doc='### Description\nThe `place_market_order` function is designed to execute a market order on a trading exchange. It takes two parameters: `symbol`, which is a string representing the stock or asset symbol (e.g., \'AAPL\' for Apple Inc.), and `qty`, which is an integer indicating the quantity of shares to be purchased or sold. The function encapsulates the trading logic necessary to place the order and returns a confirmation message indicating the number of shares ordered and the symbol associated with the transaction.\n\n### Context\nThis function should be used in scenarios where a user or a trading application needs to execute a market order for a specific asset. It is particularly useful in automated trading systems, trading bots, or any application that requires real-time trading capabilities. Users should ensure that they have sufficient funds and that the market is open before invoking this function to avoid error

In [89]:
registry.save_all_tools_to_db()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [90]:
registry.tools.keys()

dict_keys(['place_market_order', 'get_weather', 'get_news', 'get_stock_price', 'get_recipe'])

In [91]:
tools = memory_manager.read_toolbox_entry(query="market order and weather", limit=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [92]:
schemas = []
for tool in tools:
    tool_info = registry.tools.get(tool)
    if tool_info:
        schemas.append(tool_info["schema"])

In [93]:
schemas

[{'type': 'function',
  'function': {'name': 'get_weather',
   'strict': True,
   'parameters': {'properties': {'location': {'title': 'Location',
      'type': 'string'}},
    'required': ['location'],
    'title': 'WeatherParams',
    'type': 'object',
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'place_market_order',
   'strict': True,
   'parameters': {'properties': {'symbol': {'title': 'Symbol',
      'type': 'string'},
     'qty': {'title': 'Qty', 'type': 'integer'}},
    'required': ['symbol', 'qty'],
    'title': 'OrderParams',
    'type': 'object',
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_news',
   'strict': True,
   'parameters': {'properties': {'topic': {'title': 'Topic',
      'type': 'string'}},
    'required': ['topic'],
    'title': 'NewsParams',
    'type': 'object',
    'additionalProperties': False}}}]

In [97]:
res=openai_client.generate(
    messages=[{"role": "user", "content": "What's the weather in New York?"}],
    tools=schemas,
    tool_choice="auto",
    return_full_response=True
)

In [103]:
res

{'content': None,
 'text': 'No tool call content returned.',
 'tool_calls': [{'id': 'call_KF6mpX2sVtSbUn5EvmphvCbl',
   'type': 'function',
   'name': 'get_weather',
   'arguments': '{"location":"New York"}'}],
 'usage': {'prompt_tokens': 100, 'completion_tokens': 15, 'total_tokens': 115},
 'latency_ms': 2248.11,
 'model_name': 'openai/gpt-4o-mini',
 'raw_response': ModelResponse(id='chatcmpl-DU8wP9LzSLyvklNz28iCtufW2UBvP', created=1776076581, model='gpt-4o-mini-2024-07-18', object='chat.completion', system_fingerprint='fp_a7190374f3', choices=[Choices(finish_reason='tool_calls', index=0, message=Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"location":"New York"}', name='get_weather'), id='call_KF6mpX2sVtSbUn5EvmphvCbl', type='function')], function_call=None, provider_specific_fields={'refusal': None}, annotations=[]), provider_specific_fields={})], usage=Usage(completion_tokens=15, prompt_tokens=100, total_tokens=115, 

In [102]:
print("LLM Response with Tool Calls:", res)

LLM Response with Tool Calls: {'content': None, 'text': 'No tool call content returned.', 'tool_calls': [{'id': 'call_KF6mpX2sVtSbUn5EvmphvCbl', 'type': 'function', 'name': 'get_weather', 'arguments': '{"location":"New York"}'}], 'usage': {'prompt_tokens': 100, 'completion_tokens': 15, 'total_tokens': 115}, 'latency_ms': 2248.11, 'model_name': 'openai/gpt-4o-mini', 'raw_response': ModelResponse(id='chatcmpl-DU8wP9LzSLyvklNz28iCtufW2UBvP', created=1776076581, model='gpt-4o-mini-2024-07-18', object='chat.completion', system_fingerprint='fp_a7190374f3', choices=[Choices(finish_reason='tool_calls', index=0, message=Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"location":"New York"}', name='get_weather'), id='call_KF6mpX2sVtSbUn5EvmphvCbl', type='function')], function_call=None, provider_specific_fields={'refusal': None}, annotations=[]), provider_specific_fields={})], usage=Usage(completion_tokens=15, prompt_tokens=100, tot

In [104]:
storage_manager.get_toolbox_table().search(
            query="market order and weather",
            query_type="vector",
            vector_column_name="vector",
        ).distance_type("cosine").limit(10).to_pandas().sort_values("_distance")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,id,thread_id,text,vector,metadata,created_at,_distance
0,NaN,NaN,Tool: get_weather\n\nDescription: ### Descript...,"[-0.024518732, 0.033088613, 0.6567343, 0.25223...","[{'key': 'tool_name', 'value': 'get_weather'}]",2026-04-13 10:31:35.904428,0.409622
1,NaN,NaN,Tool: place_market_order\n\nDescription: ### D...,"[0.03776844, 0.09925843, 0.28146562, -0.194913...","[{'key': 'tool_name', 'value': 'place_market_o...",2026-04-13 10:31:34.103340,0.450017
2,NaN,NaN,Tool: get_news\n\nDescription: ### Description...,"[0.13725394, 0.12182571, 0.4648163, 0.12863404...","[{'key': 'tool_name', 'value': 'get_news'}]",2026-04-13 10:31:37.426049,0.457614
3,NaN,NaN,Tool: get_stock_price\n\nDescription: ### Desc...,"[-0.094043404, 0.082601756, 0.2684411, 0.12457...","[{'key': 'tool_name', 'value': 'get_stock_pric...",2026-04-13 10:31:38.877785,0.485027
4,NaN,NaN,Tool: get_recipe\n\nDescription: ### Descripti...,"[0.10440569, 0.21677394, 0.38567495, -0.210963...","[{'key': 'tool_name', 'value': 'get_recipe'}]",2026-04-13 10:31:40.417841,0.588484


'{"name": "place_market_order", "schema": {"type": "function", "function": {"name": "OrderParams", "strict": true, "parameters": {"properties": {"symbol": {"title": "Symbol", "type": "string"}, "qty": {"title": "Qty", "type": "integer"}}, "required": ["symbol", "qty"], "title": "OrderParams", "type": "object", "additionalProperties": false}}}, "augmented_metadata": {"search_doc": "### Description\\nThe `place_market_order` function is designed to execute a market order on a trading exchange. It takes two parameters: `symbol`, which represents the stock or asset to be traded, and `qty`, which indicates the quantity of shares to be purchased or sold. The function encapsulates the trading logic necessary to place the order and returns a confirmation message indicating the number of shares ordered and the symbol of the asset.\\n\\n### Context\\nThis function should be used in scenarios where a user or a trading application needs to execute a market order for a specific asset. It is particularly useful in automated trading systems, trading bots, or any application that requires real-time trading capabilities. The function is part of a larger trading framework that likely includes other order types and trading functionalities.\\n\\n### Trigger Queries\\n1. \\"How do I place a market order for 100 shares of AAPL?\\"\\n2. \\"Can you execute a market order for 50 shares of TSLA?\\"\\n3. \\"What is the command to buy 200 shares of GOOGL at market price?\\"\\n4. \\"I need to place a market order for 10 shares of AMZN, how do I do that?\\"\\n5. \\"What function do I use to sell 75 shares of MSFT on the exchange?\\"", "trigger_queries": ["How do I place a market order for 100 shares of AAPL?", "Can you execute a market order for 50 shares of TSLA?", "What is the command to buy 200 shares of GOOGL at market price?", "I need to place a market order for 10 shares of AMZN, how do I do that?", "What function do I use to sell 75 shares of MSFT on the exchange?"]}}'

In [44]:
sample_data = [
    KnowledgeBaseEntry(thread_id="thread1", text="This is a knowledge base entry.", vector=None),

    # add another about USA and its capital
    KnowledgeBaseEntry(thread_id="thread3", text="The capital of USA is Washington D.C.", vector=None),
    KnowledgeBaseEntry(thread_id="thread4", text="The capital of France is Paris.", vector=None),
    # food related
    KnowledgeBaseEntry(thread_id="thread5", text="Pizza is a popular Italian dish.", vector=None),
    KnowledgeBaseEntry(thread_id="thread6", text="Sushi is a traditional Japanese dish.", vector=None),
    # maths related
    KnowledgeBaseEntry(thread_id="thread7", text="The Pythagorean theorem states that a^2 + b^2 = c^2.", vector=None),
    KnowledgeBaseEntry(thread_id="thread8", text="The derivative of sin(x) is cos(x).", vector=None),
    # Climate related
    KnowledgeBaseEntry(thread_id="thread9", text="Climate change is caused by human activities.", vector=None),
    KnowledgeBaseEntry(thread_id="thread10", text="Renewable energy sources include solar and wind power.", vector=None)
]

memory_manager.add_knowledge_base_entry(sample_data)

In [51]:
storage_manager.get_knowledge_base_table().create_fts_index("text")  # Create a fts index before the hybrid search

In [44]:
storage_manager.get_knowledge_base_table().search(
    query="capital of USA",
    query_type="vector",
    vector_column_name="vector",
).to_pandas().sort_values("_distance").head(5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,id,thread_id,text,vector,metadata,created_at,_distance


In [116]:
tem=storage_manager.get_toolbox_table().search(
            query="market order and weather",
            query_type="vector",
            vector_column_name="vector",
        ).distance_type("cosine").to_pandas().sort_values("_distance").head(5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
linear.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [129]:
f"{tem["text"].tolist()}"

'[\'Tool: get_weather\\n\\nDescription: ### Description\\nThe `get_weather` function is designed to fetch and return the current weather conditions for a specified location. It takes a single parameter, `location`, which is a string representing the name of the place for which the weather information is requested. The function currently simulates a weather response by returning a hardcoded string indicating that the weather is sunny and 25°C for the given location. This function serves as a placeholder for more complex weather-fetching logic that could involve API calls to a weather service.\\n\\n### Context\\nThis function should be used in applications where users need to retrieve current weather information based on a specific location. It is particularly useful in scenarios such as:\\n- Weather applications that provide users with real-time weather updates.\\n- Travel planning tools that help users understand the weather conditions at their destination.\\n- Event planning applicati

In [115]:
type(tem[0][0])

dict

In [14]:
# create sample conversation logs
conversation_logs = [
    ConversationLog(thread_id="thread1", role="user", content="Hello, how are you?"),
    ConversationLog(thread_id="thread1", role="assistant", content="I'm good, thank you! How can I assist you today?"),
    ConversationLog(thread_id="thread1", role="user", content="Can you tell me a joke?"),
    ConversationLog(thread_id="thread1", role="assistant", content="Sure! Why don't scientists trust atoms? Because they make up everything!"),
    ConversationLog(thread_id="thread2", role="user", content="What's the weather like today?"),
    ConversationLog(thread_id="thread2", role="assistant", content="It's sunny and warm today!"),
]

memory_manager.add_conversation_log(conversation_logs)

In [16]:
storage_manager.get_conversation_table().search().where("thread_id = 'thread1'").to_pandas().sort_values("timestamp")

,id,thread_id,role,content,timestamp,metadata,created_at,summary_id
0,NaN,thread1,user,"Hello, how are you?",2026-04-14 20:34:09.558827,NaN,2026-04-14 20:34:09.558828,NaN
1,NaN,thread1,assistant,"I'm good, thank you! How can I assist you today?",2026-04-14 20:34:09.558833,NaN,2026-04-14 20:34:09.558834,NaN
2,NaN,thread1,user,Can you tell me a joke?,2026-04-14 20:34:09.558835,NaN,2026-04-14 20:34:09.558835,NaN
3,NaN,thread1,assistant,Sure! Why don't scientists trust atoms? Becaus...,2026-04-14 20:34:09.558836,NaN,2026-04-14 20:34:09.558837,NaN


In [1]:
AGENT_SYSTEM_PROMPT = """
# Role
You are a memory-aware agentic assistant with access to tools.

# Context Window Structure (Partitioned Segments)
The user input is a partitioned context window. It contains a `# Question` section followed by memory segments.
Treat each segment as a distinct memory store with a specific purpose:
- `## Conversation Memory`
- `## Knowledge Base Memory`
- `## Workflow Memory`
- `## Entity Memory`
- `## Summary Memory`

# Memory Store Semantics
- Conversation Memory: Recent thread-level dialogue and instructions. Use it for continuity, user preferences, and unresolved requests.
- Knowledge Base Memory: Retrieved documents/passages. Use it to ground factual and technical claims.
- Workflow Memory: Prior execution patterns and step sequences. Use it to plan tool usage; adapt patterns, do not copy blindly.
- Entity Memory: Named people/orgs/systems and descriptors. Use it to disambiguate references and keep naming consistent.
- Summary Memory: Compressed older context represented by summary IDs. When thread-scoped summaries exist, prefer summaries for the active thread_id.

# Summary Expansion Policy
If critical detail is only present in Summary Memory or appears ambiguous, call `expand_summary(summary_id)` before relying on it.

# Operating Rules
1. Start with the provided memory segments before using tools.
2. If segments conflict, prioritize: current `# Question` > latest Conversation Memory > Knowledge Base evidence > older summaries/workflows.
3. Use only the tools provided in this turn and choose the minimum necessary tool calls.
4. If memory is insufficient, state what is missing and then use an appropriate tool.
5. For conversation compaction, use `summarize_and_store` with `thread_id` so source conversation units are marked as summarized.
"""

In [ ]:
def call_agent(query:str, thread_id:str="1", max_iteration=10):

    memory_context = ""
    memory_context = memory_manager.read_conversation_logs(thread_id=thread_id) + "\n\n"
    memory_context += memory_manager.read_knowledge_base_entry(query=query) + "\n\n"
    memory_context += memory_manager.read_workflow_entry(query=query) + "\n\n"

    workflow_steps=[]


    memory_manager.add_conversation_log([ConversationLog(thread_id=thread_id, role="user", content=query)])
    memory_manager.add_entity_entry(query)

    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": f"# Question\n{query}\n\n"}
    ]

    tools = memory_manager.read_toolbox_entry(query=query, limit=5)

    for iteration in range(max_iteration):
        print(f"\n\n=== Iteration {iteration+1} ===")
        print("Current Memory Context:", memory_context)
        print("Available Tools:", tools)

        tool_schemas = []
        for tool in tools:
            tool_info = registry.tools.get(tool)
            if tool_info:
                tool_schemas.append(tool_info["schema"])

        response = llm_client.generate(
            messages=messages,
            tools=tool_schemas,
            tool_choice="auto",
            return_full_response=True
        )

        print("LLM Response:", response)

        # Process tool calls if any
        tool_calls = response.get("tool_calls", [])
        for tool_call in tool_calls:
            tool_name = tool_call.get("name")
            arguments = tool_call.get("arguments", {})
            print(f"Calling Tool: {tool_name} with arguments {arguments}")

            # Find the corresponding function and param model
            tool_info = registry.tools.get(tool_name)
            if not tool_info:
                print(f"Tool {tool_name} not found in registry.")
                continue

            func = tool_info["callable"]
            param_model = tool_info["param_model"]

            try:
                # Validate and parse arguments using the param model
                params = param_model.model_validate(arguments)
                result = func(**params.model_dump())
                print(f"Tool Result: {result}")

                # Log the tool call and result
                memory_manager.add_tool_log([ToolLog(
                    thread_id=thread_id,
                    tool_call_id=tool_call.get("id"),
                    tool_name=tool_name,
                    tool_args=json.dumps(arguments),
                    result=str(result),
                    result_preview=str(result)[:100]  # Store a preview of the result
                )])

                # Add the tool result back into the conversation context for the next iteration
                messages.append({"role": "assistant", "content": f"Tool {tool_name} returned: {result}"})
                memory_context += f"\nAssistant used {tool_name} and got: {result}"
                workflow_steps.append(f"Used {tool_name} with args {arguments} and got result {result}")

            except Exception as e:
                print(f"Error calling tool {tool_name}: {e}")
                memory_manager.add_tool_log([ToolLog(
                    thread_id=thread_id,
                    tool_call_id=tool_call.get("id"),
                    tool_name=tool_name,
                    tool_args=json.dumps(arguments),
                    status="error",
                    error_message=str(e)
                )])

        # If no tool calls, we assume the LLM has generated a final answer or needs more information

        if workflow_steps:
            print("Workflow Steps so far:")
            for step in workflow_steps:
                print("- ", step)

        memory_manager.add_conversation_log([ConversationLog(thread_id=thread_id, role="assistant", content=response.get("content", ""))])


